# Module 8: Generative Models and the Free Energy Principle

**Spinning Up in Active Inference** | Part 3: The Active Inference Track

---

## The Transition

Welcome to Part 3. Everything changes here.

In Modules 1–7, we followed the RL path: agents learn from rewards, building value functions and policies through trial and error. Now we take a fundamentally different starting point.

**Active Inference begins not with reward, but with a model of the world.**

## Historical Context

In the 1860s, Hermann von Helmholtz proposed that perception is **unconscious inference**: the brain doesn't passively receive sensations, it actively *interprets* them by inferring their hidden causes. You don't see photons—you infer objects.

This idea lay dormant for over a century. Then in 1995, Peter Dayan and colleagues formalized it: the brain maintains a **generative model** $P(o, s) = P(o|s)P(s)$ that describes how hidden states $s$ generate observations $o$. Perception is Bayesian inference: computing $P(s|o)$.

In 2005–2010, Karl Friston took this further with the **Free Energy Principle**: biological systems minimize a quantity called **variational free energy**, which bounds the surprise of their sensory observations. This single principle unifies perception (update beliefs), action (change the world), and learning (update the model).

### What You'll Learn

1. Generative models: $P(o, s) = P(o|s)P(s)$
2. Variational inference: approximating intractable posteriors
3. The variational free energy $F$ and its decomposition
4. The Free Energy Principle: perception and action as free energy minimization
5. Building your first generative model with PGMax-AIF

## 8.1 The Bayesian Brain: Generative Models

A **generative model** specifies a joint probability over observations and hidden states:

$$P(o, s) = P(o|s) \cdot P(s)$$

- $P(o|s)$: the **likelihood** — "If the world is in state $s$, what observations do I expect?"
- $P(s)$: the **prior** — "Before observing anything, what do I believe about the world?"

The agent's goal is to invert this model: given an observation $o$, infer the most likely hidden state $s$.

By Bayes' rule:

$$P(s|o) = \frac{P(o|s) P(s)}{P(o)} = \frac{P(o|s) P(s)}{\sum_{s'} P(o|s') P(s')}$$

The denominator $P(o) = \sum_{s'} P(o|s')P(s')$ is the **model evidence** (or marginal likelihood). It tells us how well the entire model explains the observation. Its negative log, $-\ln P(o)$, is called **surprisal** or **self-information**.

### Example: A Weather Model

- Hidden states: {Sunny, Cloudy, Rainy}
- Observations: {Bright, Dim, Wet}
- $P(\text{Bright}|\text{Sunny}) = 0.9$, $P(\text{Bright}|\text{Cloudy}) = 0.3$, etc.

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')

from alf.generative_model import GenerativeModel
from alf.free_energy import (
    variational_free_energy,
    free_energy_from_beliefs,
    expected_free_energy_decomposed,
    EFEDecomposition,
)

from utils.plotting import (plot_matrix_heatmap, plot_belief_evolution,
                            plot_efe_decomposition, dual_perspective_box,
                            plot_policy_distribution, COLORS)

np.random.seed(42)
print("Module 8: Generative Models and the Free Energy Principle")
print("="*58)

In [ ]:
# Build a simple generative model: 3 states, 3 observations
# This is the A-matrix (likelihood): P(o|s)
# Rows = observations, Columns = states

state_names = ['Sunny', 'Cloudy', 'Rainy']
obs_names = ['Bright', 'Dim', 'Wet']

# A[o, s] = P(observation=o | state=s)
A = np.array([
    [0.9, 0.3, 0.05],   # P(Bright | Sunny)=0.9, P(Bright | Cloudy)=0.3, ...
    [0.08, 0.6, 0.15],  # P(Dim | ...)
    [0.02, 0.1, 0.8],   # P(Wet | ...)
])

# Verify columns sum to 1 (each state generates a valid distribution over obs)
print("Column sums (should be 1.0):", A.sum(axis=0))

# Prior: P(s) - uniform (no prior preference)
prior = np.array([1/3, 1/3, 1/3])

# Visualize the likelihood matrix
fig, ax = plt.subplots(1, 1, figsize=(6, 5))
plot_matrix_heatmap(A, title="Likelihood Matrix A: P(observation | state)",
                    row_labels=obs_names, col_labels=state_names,
                    cmap='YlOrRd', ax=ax)
ax.set_xlabel("Hidden State")
ax.set_ylabel("Observation")
plt.tight_layout()
plt.show()

print("\nReading the A-matrix:")
print("  If it's Sunny, I'll observe Bright with P=0.90")
print("  If it's Rainy, I'll observe Wet with P=0.80")
print("  If it's Cloudy, I'll observe Dim with P=0.60")

## 8.2 Bayesian Inference: Computing the Posterior

Given an observation, we can compute the exact posterior using Bayes' rule:

$$P(s|o) = \frac{P(o|s) P(s)}{\sum_{s'} P(o|s') P(s')}$$

For this small example, exact inference is easy. But for large state spaces, the sum in the denominator becomes intractable. This is where **variational inference** comes in.

In [ ]:
# Exact Bayesian inference
def exact_posterior(A, prior, observation):
    """Compute exact posterior P(s|o) via Bayes' rule."""
    likelihood = A[observation, :]  # P(o|s) for each state
    joint = likelihood * prior       # P(o|s) * P(s)
    evidence = joint.sum()           # P(o) = sum_s P(o,s)
    posterior = joint / evidence      # P(s|o)
    return posterior, evidence


# Observe "Bright" (obs index 0)
obs = 0  # Bright
posterior, evidence = exact_posterior(A, prior, obs)

print(f"Observation: {obs_names[obs]}")
print(f"Prior: {dict(zip(state_names, prior.round(3)))}")
print(f"Posterior: {dict(zip(state_names, posterior.round(3)))}")
print(f"Evidence P(o): {evidence:.4f}")
print(f"Surprisal -ln P(o): {-np.log(evidence):.4f} nats")

# Try all observations
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for o_idx in range(3):
    post, evid = exact_posterior(A, prior, o_idx)
    colors_bar = [COLORS['highlight'] if p == max(post) else COLORS['neutral']
                  for p in post]
    axes[o_idx].bar(state_names, post, color=colors_bar, alpha=0.85, edgecolor='black')
    axes[o_idx].set_ylim(0, 1)
    axes[o_idx].set_ylabel("P(state | obs)")
    axes[o_idx].set_title(f"Observe: {obs_names[o_idx]}\n"
                          f"Surprisal: {-np.log(evid):.2f} nats")
    axes[o_idx].grid(True, alpha=0.3, axis='y')

plt.suptitle("Bayesian Inference: Posterior Given Different Observations",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nObserving 'Bright' strongly implies Sunny (as expected).")
print("Observing 'Wet' strongly implies Rainy.")
print("Observing 'Dim' points to Cloudy, but with more uncertainty.")

## 8.3 Variational Inference: When Exact Inference Fails

For real problems, computing $P(s|o)$ exactly is **intractable** because:
- The state space is too large (e.g., millions of possible world configurations)
- The generative model has complex dependencies

**Variational inference** replaces exact computation with optimization. We introduce an **approximate posterior** $q(s)$ and adjust it to be as close as possible to the true posterior $P(s|o)$.

The measure of closeness is the **KL divergence**:

$$\text{KL}[q(s) \| P(s|o)] = \sum_s q(s) \ln \frac{q(s)}{P(s|o)} \geq 0$$

But we can't compute this directly (it requires $P(s|o)$, which is what we're trying to find!). Instead, we rearrange:

$$\text{KL}[q(s) \| P(s|o)] = \underbrace{\sum_s q(s) \left[\ln q(s) - \ln P(o,s)\right]}_{\text{Variational Free Energy } F} + \ln P(o)$$

Since $\ln P(o)$ doesn't depend on $q$, **minimizing $F$ is equivalent to minimizing the KL divergence**.

## 8.4 The Variational Free Energy

The **Variational Free Energy** (VFE) is the central quantity in Active Inference:

$$\boxed{F = \sum_s q(s) \left[\ln q(s) - \ln P(o|s) - \ln P(s)\right]}$$

This can be decomposed in two illuminating ways:

### Decomposition 1: KL + Evidence

$$F = \text{KL}[q(s) \| P(s|o)] - \ln P(o)$$

Since KL $\geq 0$:

$$\boxed{F \geq -\ln P(o)}$$

**Free energy is an upper bound on surprisal.** Minimizing $F$ minimizes surprise.

### Decomposition 2: Complexity - Accuracy (ELBO)

$$F = \underbrace{\text{KL}[q(s) \| P(s)]}_{\text{Complexity}} - \underbrace{\mathbb{E}_q[\ln P(o|s)]}_{\text{Accuracy}}$$

- **Complexity**: How far has $q(s)$ moved from the prior $P(s)$? (Penalizes overfitting)
- **Accuracy**: How well does $q(s)$ explain the observation? (Rewards good explanations)

Good inference balances these: find beliefs that explain the data well without straying too far from prior expectations.

In [ ]:
# Compute VFE for different beliefs and show it's minimized at the true posterior

# Observe "Bright" (obs=0)
obs = 0
true_posterior, evidence = exact_posterior(A, prior, obs)
true_surprisal = -np.log(evidence)

print(f"True posterior P(s|'{obs_names[obs]}'): {true_posterior.round(4)}")
print(f"True surprisal -ln P(o): {true_surprisal:.4f} nats")
print()

# Sweep q(s) = [p, (1-p)/2, (1-p)/2] — varying belief about 'Sunny'
p_range = np.linspace(0.01, 0.99, 100)
F_values = []

for p in p_range:
    q = np.array([p, (1-p)/2, (1-p)/2])
    F = variational_free_energy(q, A, prior, obs)
    F_values.append(F)

# Also compute F at the true posterior
F_at_true = variational_free_energy(true_posterior, A, prior, obs)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(p_range, F_values, color=COLORS['aif'], linewidth=2,
             label='F(q)')
axes[0].axhline(y=true_surprisal, color=COLORS['danger'], linestyle='--',
                linewidth=1.5, label=f'-ln P(o) = {true_surprisal:.3f}')
axes[0].axvline(x=true_posterior[0], color=COLORS['highlight'], linestyle=':',
                linewidth=1.5, label=f'True P(Sunny|Bright) = {true_posterior[0]:.3f}')
axes[0].scatter([true_posterior[0]], [F_at_true], color=COLORS['highlight'],
                s=100, zorder=5, edgecolors='black')
axes[0].set_xlabel('q(Sunny)')
axes[0].set_ylabel('Free Energy F (nats)')
axes[0].set_title('VFE is Minimized at the True Posterior')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)

# Annotate the gap = KL divergence
mid_p = 0.5
q_mid = np.array([mid_p, (1-mid_p)/2, (1-mid_p)/2])
F_mid = variational_free_energy(q_mid, A, prior, obs)
axes[0].annotate('', xy=(mid_p, true_surprisal), xytext=(mid_p, F_mid),
                 arrowprops=dict(arrowstyle='<->', color=COLORS['epistemic'], lw=2))
axes[0].text(mid_p + 0.03, (F_mid + true_surprisal)/2,
             'KL[q||P(s|o)]', fontsize=10, color=COLORS['epistemic'])

# Decomposition: Complexity vs Accuracy
complexity_values = []
accuracy_values = []

for p in p_range:
    q = np.array([p, (1-p)/2, (1-p)/2])
    # KL[q || prior]
    kl = np.sum(q * np.log(np.clip(q, 1e-16, None) / np.clip(prior, 1e-16, None)))
    # E_q[ln P(o|s)]
    accuracy = np.sum(q * np.log(np.clip(A[obs, :], 1e-16, None)))
    complexity_values.append(kl)
    accuracy_values.append(-accuracy)  # Negative accuracy (to show as cost)

axes[1].plot(p_range, complexity_values, color=COLORS['danger'], linewidth=2,
             label='Complexity: KL[q||prior]')
axes[1].plot(p_range, accuracy_values, color=COLORS['reward'], linewidth=2,
             label='Neg. Accuracy: -E_q[ln P(o|s)]')
axes[1].plot(p_range, F_values, color=COLORS['aif'], linewidth=2.5,
             linestyle='--', label='F = Complexity - Accuracy')
axes[1].axvline(x=true_posterior[0], color=COLORS['highlight'], linestyle=':',
                linewidth=1.5)
axes[1].set_xlabel('q(Sunny)')
axes[1].set_ylabel('Free Energy Components (nats)')
axes[1].set_title('ELBO Decomposition: Complexity vs Accuracy')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)

plt.suptitle("Variational Free Energy", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nF at true posterior: {F_at_true:.4f} nats")
print(f"-ln P(o):           {true_surprisal:.4f} nats")
print(f"Gap (= KL at min):  {F_at_true - true_surprisal:.6f} nats")
print("\nAt the true posterior, F equals -ln P(o). The KL gap is zero.")

## 8.5 How F Changes with Observations: Surprise

The Free Energy Principle says organisms minimize free energy. Surprising observations (those poorly predicted by the generative model) increase $F$, motivating the agent to either:

1. **Update beliefs** (perception): change $q(s)$ to better explain $o$ $\rightarrow$ reduces $F$
2. **Take action** (active inference): change $o$ by acting on the world $\rightarrow$ reduces $F$

Let's see how the free energy landscape changes with different observations.

In [ ]:
# Show how F changes with observation
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for o_idx in range(3):
    F_vals = []
    for p in p_range:
        q = np.array([p, (1-p)/2, (1-p)/2])
        F = variational_free_energy(q, A, prior, o_idx)
        F_vals.append(F)

    posterior_o, evidence_o = exact_posterior(A, prior, o_idx)
    surprisal_o = -np.log(evidence_o)
    F_min = variational_free_energy(posterior_o, A, prior, o_idx)

    axes[o_idx].plot(p_range, F_vals, color=COLORS['aif'], linewidth=2)
    axes[o_idx].axhline(y=surprisal_o, color=COLORS['danger'], linestyle='--',
                        alpha=0.7, label=f'Surprisal = {surprisal_o:.2f}')
    axes[o_idx].scatter([posterior_o[0]], [F_min], color=COLORS['highlight'],
                        s=100, zorder=5, edgecolors='black',
                        label=f'Min F = {F_min:.2f}')
    axes[o_idx].set_xlabel('q(Sunny)')
    axes[o_idx].set_ylabel('F (nats)')
    axes[o_idx].set_title(f'Observe: {obs_names[o_idx]}')
    axes[o_idx].legend(fontsize=8)
    axes[o_idx].grid(True, alpha=0.3)

plt.suptitle("Free Energy Landscape Changes with Observation",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Compare surprisal across observations
surprisals = []
for o_idx in range(3):
    _, evid = exact_posterior(A, prior, o_idx)
    surprisals.append(-np.log(evid))
    print(f"Observation '{obs_names[o_idx]}': surprisal = {surprisals[-1]:.3f} nats")

print(f"\nMost surprising: '{obs_names[np.argmax(surprisals)]}' "
      f"(surprisal = {max(surprisals):.3f})")
print(f"Least surprising: '{obs_names[np.argmin(surprisals)]}' "
      f"(surprisal = {min(surprisals):.3f})")

## 8.6 The Free Energy Principle

Karl Friston's **Free Energy Principle** (FEP) makes a bold claim:

> *Any self-organizing system that persists in the face of environmental change must minimize variational free energy.*

This provides a unified account of:

| Process | How it minimizes F | Mechanism |
|---|---|---|
| **Perception** | Update $q(s)$ to explain observations | Belief updating (inference) |
| **Action** | Change $o$ to match predictions/preferences | Motor control |
| **Learning** | Update model parameters ($A$, $B$, etc.) | Synaptic plasticity |
| **Attention** | Adjust precision of beliefs | Gain modulation |
| **Planning** | Evaluate future policies by expected F | Active inference |

### Perception = Inference

When you receive a new observation, you update your beliefs $q(s)$ to minimize $F$. This is equivalent to approximating the Bayesian posterior $P(s|o)$.

### Action = Control

When your beliefs predict certain observations (preferred observations, encoded in $C$), but the actual observations don't match, you ACT to change the world so that observations match your predictions. This is **active inference**: you make the world conform to your model, rather than just passively updating your model.

In [ ]:
# Demonstrate perception as free energy minimization
# Start with a uniform belief and iteratively update toward the posterior

def gradient_descent_vfe(A, prior, obs, n_steps=50, lr=0.5):
    """Minimize VFE by gradient descent on the belief simplex."""
    n_states = len(prior)
    # Start with uniform belief
    q = np.ones(n_states) / n_states
    history = [q.copy()]
    F_history = [variational_free_energy(q, A, prior, obs)]

    for step in range(n_steps):
        # Compute gradient of F w.r.t. log q (natural parameters)
        # dF/d(log q_s) = q_s * [ln q_s - ln P(o|s) - ln P(s) + 1]
        # Simplified: update in the direction of the likelihood
        log_likelihood = np.log(np.clip(A[obs, :], 1e-16, None))
        log_prior = np.log(np.clip(prior, 1e-16, None))
        log_q = np.log(np.clip(q, 1e-16, None))

        # Gradient in natural parameters: move toward posterior
        target = np.exp(log_likelihood + log_prior)
        target = target / target.sum()

        # Soft update
        q = (1 - lr) * q + lr * target
        q = np.clip(q, 1e-10, None)
        q = q / q.sum()

        history.append(q.copy())
        F_history.append(variational_free_energy(q, A, prior, obs))

    return history, F_history


# Run perception (belief updating) for "Wet" observation
obs = 2  # Wet
belief_history, F_history = gradient_descent_vfe(A, prior, obs, n_steps=20, lr=0.3)
true_post, _ = exact_posterior(A, prior, obs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Belief evolution
plot_belief_evolution(belief_history, state_names=state_names,
                      title=f"Perception: Beliefs Converge After Observing '{obs_names[obs]}'",
                      ax=axes[0])
# Mark true posterior
for i, name in enumerate(state_names):
    axes[0].axhline(y=true_post[i], color=plt.cm.Set2(i/3), linestyle='--', alpha=0.5)
axes[0].set_xlabel("Belief Update Step")

# Free energy decreasing
axes[1].plot(F_history, 'o-', color=COLORS['aif'], linewidth=2, markersize=5)
axes[1].axhline(y=-np.log(exact_posterior(A, prior, obs)[1]),
                color=COLORS['danger'], linestyle='--',
                label='Lower bound (-ln P(o))')
axes[1].set_xlabel("Belief Update Step")
axes[1].set_ylabel("Variational Free Energy (nats)")
axes[1].set_title("Free Energy Decreases During Inference")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle("Perception as Free Energy Minimization",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nInitial belief (uniform): {belief_history[0].round(3)}")
print(f"Final belief:             {belief_history[-1].round(3)}")
print(f"True posterior:            {true_post.round(3)}")
print(f"\nF decreased from {F_history[0]:.3f} to {F_history[-1]:.3f} nats")

## 8.7 Building a Full Generative Model: The POMDP Setup

Active Inference uses a **Partially Observable Markov Decision Process** (POMDP) formulation. The generative model has four key matrices:

| Matrix | Meaning | Shape | RL Equivalent |
|---|---|---|---|
| **A** | Likelihood $P(o\|s)$ | (n_obs, n_states) | Observation model |
| **B** | Transitions $P(s'\|s,a)$ | (n_states, n_states, n_actions) | $T(s'\|s,a)$ in model-based RL |
| **C** | Preferences $\ln P(o)$ | (n_obs,) | Reward function $R(o)$ |
| **D** | Prior beliefs $P(s_0)$ | (n_states,) | Initial state distribution |

Let's build a complete generative model and see how these matrices work together.

In [ ]:
# Build a complete generative model with PGMax-AIF
# Scenario: A simple foraging task
# 4 locations: Home, Field, Forest, River
# 4 observations: Safe, Food, Wood, Water
# 4 actions: Stay, Go-Field, Go-Forest, Go-River

location_names = ['Home', 'Field', 'Forest', 'River']
observation_names = ['Safe', 'Food', 'Wood', 'Water']
action_names = ['Stay', 'To Field', 'To Forest', 'To River']

n_states = 4
n_obs = 4
n_actions = 4

# A: Likelihood matrix P(o|s)
# Each location generates characteristic observations
A_forage = np.array([
    [0.80, 0.05, 0.05, 0.05],  # Safe: mostly at Home
    [0.05, 0.80, 0.10, 0.05],  # Food: mostly at Field
    [0.10, 0.05, 0.75, 0.05],  # Wood: mostly at Forest
    [0.05, 0.10, 0.10, 0.85],  # Water: mostly at River
])

# B: Transition model P(s'|s,a)
# Each action moves to the corresponding location (mostly)
B_forage = np.zeros((n_states, n_states, n_actions))

# Stay action: remain in current state
B_forage[:, :, 0] = np.eye(n_states)

# Go-Field: from anywhere, 90% chance of reaching Field
for s in range(n_states):
    B_forage[1, s, 1] = 0.9   # Reach Field
    B_forage[s, s, 1] = 0.1   # Stay put (fail to move)

# Go-Forest: 90% chance of reaching Forest
for s in range(n_states):
    B_forage[2, s, 2] = 0.9
    B_forage[s, s, 2] = 0.1

# Go-River: 90% chance of reaching River
for s in range(n_states):
    B_forage[3, s, 3] = 0.9
    B_forage[s, s, 3] = 0.1

# C: Preferences (log-preferences over observations)
# Prefer Food, like Water, neutral about others
C_forage = np.log(np.array([0.05, 0.60, 0.10, 0.25]) + 1e-16)

# D: Prior (start at Home)
D_forage = np.array([0.9, 0.03, 0.03, 0.04])

# Create GenerativeModel
gm = GenerativeModel(
    A=[A_forage],
    B=[B_forage],
    C=[C_forage],
    D=[D_forage],
    T=1
)

print(f"Generative Model Created:")
print(f"  States:      {n_states} ({', '.join(location_names)})")
print(f"  Observations:{n_obs} ({', '.join(observation_names)})")
print(f"  Actions:     {n_actions} ({', '.join(action_names)})")
print(f"  Policies:    {gm.num_policies}")
print(f"  Horizon:     {gm.T} step(s)")

In [ ]:
# Visualize all four matrices
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# A: Likelihood
plot_matrix_heatmap(A_forage, title="A: Likelihood P(observation | location)",
                    row_labels=observation_names, col_labels=location_names,
                    cmap='YlOrRd', ax=axes[0, 0])
axes[0, 0].set_xlabel("Hidden State (Location)")
axes[0, 0].set_ylabel("Observation")

# B: Transition (for 'Go Field' action)
plot_matrix_heatmap(B_forage[:, :, 1], title="B: Transitions P(s'|s, 'Go Field')",
                    row_labels=location_names, col_labels=location_names,
                    cmap='Blues', ax=axes[0, 1])
axes[0, 1].set_xlabel("Current Location")
axes[0, 1].set_ylabel("Next Location")

# C: Preferences
colors_pref = [COLORS['aif'] if c == max(C_forage) else COLORS['neutral']
               for c in C_forage]
axes[1, 0].bar(observation_names, np.exp(C_forage), color=colors_pref,
               alpha=0.85, edgecolor='black')
axes[1, 0].set_ylabel("P(o) = exp(C)")
axes[1, 0].set_title("C: Preferences over Observations")
axes[1, 0].grid(True, alpha=0.3, axis='y')

# D: Prior
colors_prior = [COLORS['highlight'] if d == max(D_forage) else COLORS['neutral']
                for d in D_forage]
axes[1, 1].bar(location_names, D_forage, color=colors_prior,
               alpha=0.85, edgecolor='black')
axes[1, 1].set_ylabel("P(s_0)")
axes[1, 1].set_title("D: Prior Beliefs (Initial State)")
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.suptitle("The Four Matrices of a Generative Model (A, B, C, D)",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print("\nA tells the agent what to expect (observation model)")
print("B tells the agent how actions change the world (dynamics)")
print("C tells the agent what it wants (preferences)")
print("D tells the agent where it starts (prior beliefs)")

In [ ]:
# Compute EFE for each action to see which the agent prefers
beliefs = D_forage.copy()  # Start with prior belief (at Home)

pragmatic_values = []
epistemic_values = []
G_values = []

for a in range(n_actions):
    decomp = expected_free_energy_decomposed(
        A_forage, B_forage, C_forage, beliefs, a
    )
    G_values.append(decomp.G_total)
    pragmatic_values.append(decomp.pragmatic)
    epistemic_values.append(decomp.epistemic)

    print(f"Action '{action_names[a]}':")
    print(f"  G(a) = {decomp.G_total:.3f}  "
          f"(pragmatic={decomp.pragmatic:.3f}, epistemic={decomp.epistemic:.3f})")

# Plot EFE decomposition
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

plot_efe_decomposition(pragmatic_values, epistemic_values,
                       action_names=action_names,
                       title="EFE Decomposition: Which Action to Take?",
                       ax=axes[0])

# Policy from EFE
G_arr = np.array(G_values)
gamma_precision = 4.0
policy_probs = np.exp(-gamma_precision * G_arr)
policy_probs = policy_probs / policy_probs.sum()

plot_policy_distribution(policy_probs, action_names=action_names,
                         title=f"AIF Policy (precision gamma={gamma_precision})",
                         ax=axes[1])

plt.suptitle("Active Inference: Action Selection via EFE",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nBest action: '{action_names[np.argmax(policy_probs)]}' "
      f"(P = {max(policy_probs):.3f})")
print("The agent prefers to go to the Field (where Food is), because it has")
print("the highest pragmatic value (preference satisfaction).")

In [ ]:
# Dual Perspective Box
dual_perspective_box(
    rl_text=(
        "In RL, the agent learns <b>V(s)</b> by accumulating experience. "
        "The value of a state is its expected cumulative future reward. "
        "The agent has no explicit model of the world — it learns what "
        "to do by trial and error. The reward function R(s,a) is given "
        "by the environment, not by the agent."
    ),
    aif_text=(
        "In AIF, the agent starts with a <b>generative model P(o,s)</b> and "
        "performs inference to determine P(s|o). The 'value' of a state is "
        "how well it explains observations (accuracy) without deviating from "
        "prior expectations (complexity). Preferences (C) play the role of "
        "reward, but are specified as part of the agent's model, not externally."
    ),
    title="Value in RL vs Free Energy in AIF"
)

## 8.8 Summary: The Free Energy Toolkit

Here are the key equations you now have in your toolkit:

### Variational Free Energy (VFE)

$$F = \sum_s q(s) \left[\ln q(s) - \ln P(o|s) - \ln P(s)\right]$$

### ELBO Decomposition

$$F = \underbrace{\text{KL}[q(s) \| P(s)]}_{\text{Complexity}} - \underbrace{\mathbb{E}_q[\ln P(o|s)]}_{\text{Accuracy}}$$

### Surprise Bound

$$F \geq -\ln P(o) \quad \text{(free energy upper-bounds surprise)}$$

### The Generative Model (POMDP)

$$P(o_{0:T}, s_{0:T} | \pi) = P(s_0) \prod_{t=0}^{T} P(o_t|s_t) \prod_{t=0}^{T-1} P(s_{t+1}|s_t, a_t)$$

In matrix notation: $A \cdot D \cdot \prod_t B_{a_t}$

These tools will power everything in Modules 9–16.

## Exercises

### Exercise 8.1: Noisy Observations (Guided)
Modify the A-matrix to make observations noisier (e.g., $P(o|s) \approx 0.4$ for the correct observation). Recompute the VFE landscape. How does increased noise affect:
- The minimum value of F?
- The sharpness of the posterior?
- The surprisal of each observation?

### Exercise 8.2: Multiple Observations (Intermediate)
Extend the model to receive TWO sequential observations. After the first observation, update beliefs using `gradient_descent_vfe`. Then use the updated beliefs as the new prior for the second observation. Show that two observations reduce uncertainty more than one.

*Hint: The posterior after observation 1 becomes the prior for observation 2.*

### Exercise 8.3: Design Your Own Generative Model (Open-ended)
Create a `GenerativeModel` for a scenario of your choice (e.g., a doctor diagnosing a patient with 5 symptoms and 3 diseases, or a robot navigating a building). Define meaningful A, B, C, D matrices. Compute the EFE for each action and explain the agent's behavior.

### Exercise 8.4: Perception-as-Inference on a Cognitive Task (Advanced)
Install [NeuroGym](https://github.com/neurogym/neurogym) (`pip install neurogym`) and create a `PerceptualDecisionMaking-v0` environment. This is the Random Dots Motion task — the most widely studied perceptual decision paradigm in neuroscience. Observe its observation structure (continuous, noisy evidence over time) and consider: what would the A-matrix look like for this task? How does the brain accumulate noisy evidence to a decision threshold? This is exactly the problem variational inference solves.

---

## Key Takeaways

1. A **generative model** $P(o,s) = P(o|s)P(s)$ specifies how hidden states generate observations
2. **Variational inference** approximates the intractable posterior by minimizing free energy
3. **VFE** $F = \text{KL}[q||P(s|o)] - \ln P(o)$ upper-bounds surprisal
4. The **Free Energy Principle**: perception (update $q$) and action (change $o$) both minimize $F$
5. The **POMDP generative model** (A, B, C, D matrices) is the foundation for Active Inference
6. RL values states by expected reward; AIF values states by how well they explain observations

## Further Reading

- Dayan, P. et al. (1995). The Helmholtz machine. *Neural Computation*, 7(5). — The Bayesian brain formalized.
- Friston, K.J. (2010). The free-energy principle: a unified brain theory? *Nature Reviews Neuroscience*, 11. — The seminal FEP review.
- Smith, R., Friston, K.J. & Whyte, C.J. (2022). A step-by-step tutorial on active inference. *Journal of Mathematical Psychology*, 107. — The clearest tutorial; our primary reference.
- Parr, T., Pezzulo, G. & Friston, K.J. (2022). *Active Inference: The Free Energy Principle in Mind, Brain, and Behavior*. MIT Press. — The definitive textbook.
- Buckley, C.L. et al. (2017). The free energy principle for action and perception: A mathematical review. *Journal of Mathematical Psychology*. — Rigorous mathematical treatment.
- Whittington, J.C.R. et al. (2024). Transformers learn gating mechanisms for working memory. *arXiv*. — Attention heads spontaneously learn frontostriatal-like input/output gating; a striking demonstration of perception-as-inference emerging in transformers.
- [NeuroGym](https://github.com/neurogym/neurogym) — 30+ Gymnasium-compatible neuroscience tasks. `PerceptualDecisionMaking-v0` and `MultiSensoryIntegration-v0` are textbook examples of Bayesian inference in the brain, directly relevant to this module.

---

*Next: Module 9 — Your First Active Inference Agent (building a T-maze agent from scratch!)*